In [ ]:
import numpy as np
import evaluate
from datasets import load_dataset, load_metric
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

# 1. Load the Standard Dataset (Your Baseline Control)
# We use CoNLL-2003 as it is the standard American/British-focused dataset mentioned in the paper.
raw_datasets = load_dataset("conll2003")
label_names = raw_datasets["train"].features["ner_tags"].feature.names

# 2. Load the Tokenizer
# We'll use roberta-base (you can scale up to roberta-large later if you have the compute)
model_checkpoint = r"C:\Users\anna-\OneDrive\Desktop\ITU\4th semester\NLP and Deep Learning\Baseline"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

# 3. Data Preprocessing & Label Alignment
# This is the step where you will eventually add your dictionary swapping in the future!
# For now, it just strictly tokenizes and aligns the BIOES/BIO labels.
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Ignore special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx]) # First token of the word
            else:
                label_ids.append(-100) # Subsequent tokens of the same word
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

# 4. Load the Pretrained Model
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

# 5. Setup Evaluation Metrics
metric = load_metric("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    # Remove ignored index (special tokens)
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

# 6. Configure Training Arguments
args = TrainingArguments(
    output_dir="./baseline-ner-roberta",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 7. Train the Baseline
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

print("Starting training of the baseline model...")
trainer.train()

Using the latest cached version of the module from C:\Users\anna-\.cache\huggingface\modules\datasets_modules\datasets\conll2003\9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98 (last modified on Tue Mar 17 14:20:35 2026) since it couldn't be found locally at conll2003., or remotely on the Hugging Face Hub.
Found cached dataset conll2003 (C:/Users/anna-/.cache/huggingface/datasets/conll2003/conll2003/1.0.0/9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98)
100%|██████████| 3/3 [00:00<00:00, 599.76it/s]
Loading cached processed dataset at C:\Users\anna-\.cache\huggingface\datasets\conll2003\conll2003\1.0.0\9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98\cache-9ffe4bce2c9a3141.arrow
Loading cached processed dataset at C:\Users\anna-\.cache\huggingface\datasets\conll2003\conll2003\1.0.0\9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98\cache-4fb826750188403d.arrow
Some weights of the model checkpoint at C:\Users\anna-\OneDrive

Starting training of the baseline model...


wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: